# Directed, Result-Focused Analyses of Vitamin D Signatures (LINCS L1000)

This notebook presents **targeted, hypothesis-driven analyses** of transcriptomic responses to vitamin D and its analogs using LINCS L1000 Level-5 data. We focus on **clear questions, compact metrics, and publishable figures**.

**Objectives**
- Define and validate a **core Vitamin D (VDR) signature** across contexts.
- Quantify responses **by cell line** and **by analog** using a single **core score**.
- Test **dose–response** monotonicity (Spearman ρ) and estimate **potency** (slope of `core_score ~ log10(dose)` with 95% CI).
- Perform **pathway enrichment** (GSEA Preranked, Enrichr) to confirm biological themes (Hallmarks/Reactome).
- Assess **analog similarity** (correlation heatmaps; pooled vs. cell-balanced) and **robustness** of conclusions (alt core sizes).

**Inputs**
- Expression matrix: genes × signatures (LINCS L1000 Level-5, z-scores).
- Metadata: signature, cell line, analog (`cmap_name`), dose, time; gene annotations.

**Output (figures & tables)**
1. Core score distributions **by cell** and **by analog**.
2. **Dose–response**: Spearman ρ (with FDR) and **potency ranking** (median slopes + 95% CI).
3. **Enrichment dot-plots** (Hallmarks/Reactome) per cell and per analog; top UP/DOWN tables.
4. **Analog similarity** heatmaps (pooled vs. balanced).
5. **Robustness** checks (alt core vs. original).

**Notebook structure**
1. Setup & integrity checks (short).
2. Core genes and **core score** definition.
3. Core score **by cell** and **by analog** (plots).
4. **Dose–response** (ρ and slopes with CI).
5. **Pathway enrichment** (GSEA/Enrichr summaries).
6. **Analog similarity** and **robustness**.
7. Key takeaways.

> All code cells are modular and short; each section ends with a brief interpretation.

---

## 1. Setup & integrity checks

### 1.1. Scientific Stack & Plotting Setup

In [ ]:
# Core scientific stack
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Statistics and modeling
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_ind
from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix, permanova
from sklearn.decomposition import PCA

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities (for scaling or decomposition if needed)
from sklearn.preprocessing import StandardScaler

# Configure plotting aesthetics
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("viridis")
%matplotlib inline

### 1.2. Data Setup

We begin by loading all project data tables into memory:  
- **Expression matrix** (genes × signatures).  
- **Signature metadata** (perturbation, dose, cell line, etc.).  
- **Compound metadata** (compound-level annotations).  
- **Cell line metadata** (cell type, lineage, disease).  
- **Gene metadata** (landmark vs. inferred, gene symbols).  

Having all tables available ensures that downstream analyses can seamlessly combine expression values with their biological and experimental context.

In [ ]:
# Define data directory and file paths
DATA_DIR = "../data/exports"

PATHS = {
    "exp":   f"{DATA_DIR}/expression_matrix_clean.parquet",   # expression matrix
    "sig":   f"{DATA_DIR}/signature_metadata_clean.csv",      # signature metadata
    "comp":  f"{DATA_DIR}/subset_compounds_meta.csv",         # compounds
    "cells": f"{DATA_DIR}/subset_cell_lines_meta.csv",        # cell lines
    "genes": f"{DATA_DIR}/subset_genes_meta.csv",             # genes
}

# Load all tables into memory
exp_matrix = pd.read_parquet(PATHS["exp"])
metadata   = pd.read_csv(PATHS["sig"])
compounds  = pd.read_csv(PATHS["comp"])
cell_lines = pd.read_csv(PATHS["cells"])
gene_info  = pd.read_csv(PATHS["genes"])

# Quick overview of dimensions
print(f"Expression matrix: {exp_matrix.shape[0]} genes × {exp_matrix.shape[1]} signatures")
print(f"Metadata rows:     {len(metadata)}")
print(f"Compounds:         {len(compounds)}")
print(f"Cell lines:        {len(cell_lines)}")
print(f"Genes:             {len(gene_info)}")

### Data Setup Conclusion

All data tables were successfully loaded:  
- Expression matrix with 12,328 genes × 258 signatures  
- 258 metadata entries  
- 12 compounds, 5 cell lines, and 12,328 genes  

The dataset is ready for downstream analyses.

---

### 1.3. Integrity Gatekeeper

Before performing directed analyses, we run a minimal integrity check to ensure that the expression matrix and metadata are fully aligned and free of basic issues.  
This step verifies:  
- Consistent signature identifiers across tables  
- No missing values or zero-variance features  
- Dose information available and usable  

In [ ]:
def minimal_gatekeeper(exp_matrix, metadata, compounds, cell_lines, gene_info, expected_n=None):
    # Identify signature ID column in metadata
    sig_id_col = next((c for c in ["sig_id", "distil_id", "signature_id", "id"] if c in metadata.columns), None)
    if sig_id_col is None:
        raise ValueError("Signature ID column not found in metadata.")
    
    # Expression–metadata alignment (same set and order of signatures)
    exp_cols = pd.Index(map(str, exp_matrix.columns))
    meta_ids = pd.Index(metadata[sig_id_col].astype(str))
    common = exp_cols.intersection(meta_ids)
    if expected_n is not None and len(common) != expected_n:
        raise AssertionError(f"Common signatures = {len(common)} (expected {expected_n}).")
    if len(exp_cols.difference(common)) or len(meta_ids.difference(common)):
        raise AssertionError("Expression and metadata do not contain the exact same signatures.")
    meta_aligned = metadata.set_index(sig_id_col).loc[exp_cols].reset_index().rename(columns={"index": sig_id_col})
    
    # Basic integrity: NA and zero variance
    if exp_matrix.isna().any().any():
        raise AssertionError("NA values found in expression matrix.")
    if (exp_matrix.var(axis=1) == 0).any():
        raise AssertionError("Zero-variance genes detected.")
    if (exp_matrix.var(axis=0) == 0).any():
        raise AssertionError("Zero-variance signatures detected.")
    
    # Dose usability: numeric and variable within groups
    dose_col = next((c for c in meta_aligned.columns if ("dose" in c.lower()) and ("unit" not in c.lower())), None)
    if dose_col is None:
        raise AssertionError("Numeric dose column not found in metadata.")
    meta_aligned["dose_value"] = pd.to_numeric(meta_aligned[dose_col], errors="coerce")
    if meta_aligned["dose_value"].isna().any():
        raise AssertionError("Non-numeric values in dose column.")
    group_keys = [k for k in ["pert_id", "cell_id"] if k in meta_aligned.columns]
    if not group_keys:
        raise AssertionError("Missing grouping keys (pert_id/cell_id).")
    var_by_group = meta_aligned.groupby(group_keys)["dose_value"].agg(lambda x: float(np.var(x, ddof=1)) if x.notna().any() else 0.0)
    if (var_by_group == 0).all():
        raise AssertionError("No within-group dose variation; dose–response analyses are not feasible.")
    
    # Referential checks against lookup tables (lightweight)
    if "pert_id" in meta_aligned.columns and "pert_id" in compounds.columns:
        missing_comp = set(meta_aligned["pert_id"]) - set(compounds["pert_id"])
        if missing_comp:
            raise AssertionError(f"Missing compound keys in 'compounds': {len(missing_comp)}.")
    if "cell_id" in meta_aligned.columns and "cell_id" in cell_lines.columns:
        missing_cells = set(meta_aligned["cell_id"]) - set(cell_lines["cell_id"])
        if missing_cells:
            raise AssertionError(f"Missing cell IDs in 'cell_lines': {len(missing_cells)}.")
    if "gene_id" in getattr(gene_info, "columns", []):
        missing_genes = set(map(str, exp_matrix.index)) - set(map(str, gene_info["gene_id"]))
        if missing_genes:
            raise AssertionError(f"Missing gene IDs in 'gene_info': {len(missing_genes)}.")
    
    summary = {
        "signatures": len(common),
        "genes": exp_matrix.shape[0],
        "dose_col": dose_col,
        "dose_min": float(meta_aligned["dose_value"].min()),
        "dose_max": float(meta_aligned["dose_value"].max()),
        "groups_with_variation": int((var_by_group > 0).sum()),
    }
    return meta_aligned, summary

# Run gatekeeper (expecting 258 signatures based on previous step)
metadata_aligned, gate_summary = minimal_gatekeeper(
    exp_matrix, metadata, compounds, cell_lines, gene_info, expected_n=258
)

print("Gatekeeper summary:", gate_summary)


### Integrity Gatekeeper Conclusion

- 258 signatures aligned with the expression matrix  
- 12,328 genes retained  
- Dose column detected: `pert_dose` (range ≈ 0.01 – 10 µM)  
- 35 compound–cell groups show within-group dose variation  

The dataset passes all minimal integrity checks and is suitable for dose–response analyses.

---

## 2. Utilities & parameters
### 2.1 Utilities (helpers used across sections)

This cell centralizes small, reusable helpers so later code stays short and readable:

- **ID↔symbol mapping**
  - `build_symbol_map(...)` → `Series` gene_id→gene_symbol (string keys)
  - `map_symbols_or_ids(...)` → maps a list of gene_ids to symbols (fallback to gene_id)
- **Ranked vectors & gene lists**
  - `make_preranked(series, sym_map)` → GSEA-Preranked table (`gene, score`) with dedup by |score|
- **Statistics**
  - `spearman_by_group(df, group_cols, x, y)` → Spearman ρ and p per group
  - `fit_slope_ols(df, x, y)` → OLS slope of `y ~ x` (via `np.polyfit`)
  - `add_fdr(df, p_col)` → Benjamini–Hochberg FDR column
  - `bootstrap_ci(values)` → 95% bootstrap CI for the median
- **Plot/output**
  - `savefig(fig, name, folder, dpi)` → save figure with consistent settings

All helpers are **framework-agnostic** (no side effects, no file writes unless you call `savefig`).


In [ ]:
# === Utils (compact, no new imports) ===

def to_str_index(idx_like) -> pd.Index:
    """Return a string Index from any index-like object."""
    return pd.Index(idx_like).astype(str)

def build_symbol_map(gene_info: pd.DataFrame,
                     symbol_cols=("gene_symbol","pr_gene_symbol","symbol"),
                     id_col="gene_id") -> pd.Series:
    """
    Build a Series mapping gene_id(str) -> gene_symbol (may contain NaN).
    Picks the first available symbol column in `symbol_cols`.
    """
    sym_col = next((c for c in symbol_cols if c in gene_info.columns), None)
    if sym_col is None or id_col not in gene_info.columns:
        # return empty mapping with no crash downstream
        return pd.Series(dtype=object)
    gi = gene_info[[id_col, sym_col]].drop_duplicates(subset=[id_col]).copy()
    gi[id_col] = gi[id_col].astype(str)
    gi = gi.set_index(id_col)[sym_col]
    gi.index = gi.index.astype(str)
    return gi

def map_symbols_or_ids(ids, sym_map: pd.Series):
    """
    Map a list/Index of gene_ids to symbols; fallback to the gene_id when missing/blank.
    Returns a list[str].
    """
    ids_str = to_str_index(ids)
    s = sym_map.reindex(ids_str).astype(object) if isinstance(sym_map, pd.Series) else pd.Series(index=ids_str, dtype=object)
    na_mask = s.isna() | (s.astype(str).str.strip() == "") | (s.astype(str).str.lower().isin(["nan","none"]))
    s.loc[na_mask] = ids_str[na_mask]
    return s.astype(str).tolist()

def make_preranked(series: pd.Series, sym_map: pd.Series) -> pd.DataFrame:
    """
    Build a two-column DataFrame ('gene','score') for GSEA Preranked.
    - Input `series`: index = gene_id (any dtype), values = score (float).
    - Deduplicates by gene keeping the entry with largest |score|.
    - Sorted by score descending.
    """
    s = series.copy()
    s.index = to_str_index(s.index)
    symbols = map_symbols_or_ids(s.index, sym_map)
    df = pd.DataFrame({"gene": symbols, "score": s.values})
    # deduplicate on gene by max |score|
    df = df.iloc[df["score"].abs().sort_values(ascending=False).index]
    df = df.drop_duplicates(subset="gene", keep="first")
    return df.sort_values("score", ascending=False).reset_index(drop=True)

def fit_slope_ols(df: pd.DataFrame, x="log_dose", y="core_score") -> float:
    """
    Return OLS slope of y ~ x using np.polyfit (requires >=2 unique x).
    """
    xvals = df[x].values
    yvals = df[y].values
    # numpy polyfit returns [slope, intercept] for deg=1
    slope = float(np.polyfit(xvals, yvals, 1)[0])
    return slope

def spearman_by_group(df: pd.DataFrame, group_cols, x="log_dose", y="core_score",
                      min_n=4, min_unique=2) -> pd.DataFrame:
    """
    Compute Spearman rho and p-value per group (vectorized loop).
    Filters groups with n<min_n or unique(x)<min_unique.
    """
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    rows = []
    for keys, sub in df.dropna(subset=[x, y]).groupby(group_cols):
        if len(sub) >= min_n and sub[x].nunique() >= min_unique:
            rho, p = stats.spearmanr(sub[x], sub[y])
            rec = {c: k for c, k in zip(group_cols, (keys if isinstance(keys, tuple) else (keys,)))}
            rec.update({"n": len(sub), "rho": float(rho), "pval": float(p)})
            rows.append(rec)
    return pd.DataFrame(rows)

def add_fdr(df: pd.DataFrame, p_col="pval", out_col="fdr_bh") -> pd.DataFrame:
    """
    Add BH-FDR column to a DataFrame with p-values in `p_col`.
    Returns the same DataFrame (for chaining).
    """
    if df is None or df.empty or p_col not in df.columns:
        return df
    df[out_col] = multipletests(df[p_col].values, method="fdr_bh")[1]
    return df

def bootstrap_ci(values, B=4000, alpha=0.05, random_state=0):
    """
    Nonparametric bootstrap CI for the median.
    Returns (lo, hi).
    """
    rng = np.random.RandomState(random_state)
    vals = np.asarray(values, dtype=float)
    if len(vals) == 0:
        return np.nan, np.nan
    if len(vals) == 1:
        return float(vals[0]), float(vals[0])
    boots = np.empty(B, dtype=float)
    for i in range(B):
        sample = rng.choice(vals, size=len(vals), replace=True)
        boots[i] = np.median(sample)
    lo, hi = np.percentile(boots, [100*alpha/2, 100*(1-alpha/2)])
    return float(lo), float(hi)

def savefig(fig, name: str, folder="../results/figures", dpi=400, transparent=False, tight=True):
    """
    Save a matplotlib figure with consistent settings.
    Usage: savefig(plt.gcf(), "core_by_cell")
    """
    out_dir = Path(folder)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{name}.png"
    if tight:
        fig.savefig(path, dpi=dpi, bbox_inches="tight", transparent=transparent)
    else:
        fig.savefig(path, dpi=dpi, transparent=transparent)
    print(f"[saved] {path}")
    return path


### 2.2. Parameters (global knobs)

Centralized configuration to keep the notebook reproducible and readable.  
If you tweak thresholds (e.g., `N_TOP`, `VOTE_MIN` or core sizes), do it here.

In [ ]:
# --- Parameters (edit here if needed) ---
SEED = 0
N_TOP = 50       # window size for per-context top/bottom genes (vote-count)
VOTE_MIN = 2     # minimum contexts to call a gene 'consensus'
CORE_UP_N = 42   # default core UP size (from previous consensus)
CORE_DN_N = 35   # default core DOWN size

FIG_DIR = "../results/figures"

### Setup, utilities, and parameters — summary

**Status**
- Data tables loaded and aligned (Level-5 z-scores and metadata available).
- Integrity checks passed (gatekeeper summary already printed).
- Reusable helpers registered (`ID↔symbol`, preranked builder, stats, savefig).
- Global parameters set (e.g., `N_TOP`, `VOTE_MIN`, `CORE_UP_N`, `CORE_DN_N`, `SEED`).

**Reproducibility**
- Keep all changes to thresholds in the *Parameters* cell only.
- Re-run cells 1–3 if you modify paths, imports, or parameters.

---

## 3) Core genes & core score — setup

We first aggregate expression **by cell line** to obtain a per-cell mean profile (genes × cells), and build a robust **ID→symbol** lookup.  
This prepares the inputs for:
- per-cell rankings,
- cross-cell consensus of genes, and
- the **core score** computation in the next step.

### 3.1 — Per-cell mean profile + symbol map


In [ ]:
# Identify signature ID column (robust to different schemas)
SIG_COL = next(c for c in ["sig_id", "distil_id", "signature_id", "id"] if c in metadata_aligned.columns)

# Mean z-score per cell line (genes × cells)
# Note: Level-5 already encodes treated vs control; we average within each cell_id.
labels = metadata_aligned.set_index(SIG_COL)["cell_id"]
expr_by_cell = exp_matrix.groupby(labels, axis=1).mean()

# Build result table and attach IDs/symbols
res_cell = expr_by_cell.copy()
res_cell.insert(0, "gene_id", res_cell.index.astype(str))

# Symbol map (gene_id[str] -> gene_symbol), using the utils helper
sym_map_cell = build_symbol_map(gene_info)
# Attach gene_symbol (fallback handled later where needed)
res_cell.insert(1, "gene_symbol",
                pd.Series(map_symbols_or_ids(res_cell["gene_id"], sym_map_cell), index=res_cell.index))

# Quick QC
print(f"Per-cell matrix: {res_cell.shape[0]} genes × {res_cell.shape[1]-2} cells")
print("Cells:", ", ".join([c for c in res_cell.columns if c not in ["gene_id","gene_symbol"]]))
display(res_cell.head(5)[["gene_symbol","gene_id"] + [c for c in res_cell.columns if c not in ["gene_id","gene_symbol"]][:3]])